# Phase 4 — Controlled 10-language pilot (Kaggle)

The controlled roster is English (`en`), German (`de`), Spanish (`es`), French (`fr`), Swahili (`sw`), Arabic (`ar`), Urdu (`ur`), Bulgarian (`bg`), Russian (`ru`), and Hindi (`hi`). This supervisor-approved comparison runs B2 (uniform replay), B5 (embedding-similarity replay), and M3 (published-interference replay) with the same seed (42) across all five frozen orders: 15 runs total. Run one order/seed session at a time and retain each output ZIP.

The approved comparison is descriptive and uses one preselected seed; it cannot establish seed-level robustness. M3 uses the published-interference matrix with the pre-specified k=5 URIEL+ imputation for Arabic. The protocol records approval for comparing B2, B5, and M3; it does not claim that Phase 3 selected a single winning prior.

Before running: attach `phase4_prior_comparison_code.zip` as a Kaggle input dataset, enable Internet and a GPU, accept the gated model terms for `meta-llama/Llama-3.2-1B`, and add a Kaggle secret named `HF_TOKEN`. This notebook loads the bundled source directly and does not clone GitHub. Save the output ZIP after each order so completed result records persist.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from zipfile import ZipFile

CODE_ARCHIVE_NAME = "phase4_prior_comparison_code.zip"
INPUT_ROOT = Path("/kaggle/input")
source_files = list(INPUT_ROOT.rglob("phase4/run_phase4.py"))
source_roots = {path.parents[1] for path in source_files}
if len(source_roots) == 1:
    REPO_ROOT = source_roots.pop()
elif source_roots:
    raise RuntimeError(f"Found multiple Phase 4 source trees: {sorted(map(str, source_roots))}")
else:
    code_archives = list(INPUT_ROOT.rglob(CODE_ARCHIVE_NAME))
    if len(code_archives) != 1:
        raise FileNotFoundError(f"Attach exactly one {CODE_ARCHIVE_NAME} Kaggle input; found: {code_archives}")
    extraction_root = Path("/kaggle/working/phase4_prior_comparison_source")
    with ZipFile(code_archives[0]) as archive:
        archive.extractall(extraction_root)
    source_files = list(extraction_root.rglob("phase4/run_phase4.py"))
    if len(source_files) != 1:
        raise FileNotFoundError(f"Expected one bundled phase4/run_phase4.py; found: {source_files}")
    REPO_ROOT = source_files[0].parents[1]
if not (REPO_ROOT / "phase4" / "configs" / "controlled_10lang_prior_comparison_protocol.json").is_file():
    raise FileNotFoundError("The attached code bundle does not contain the approved Phase 4 protocol.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "phase2" / "requirements-phase2.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "urielplus==1.3.3"], check=True)
sys.path.insert(0, str(REPO_ROOT))

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token
print({"source": str(REPO_ROOT), "code_archive": CODE_ARCHIVE_NAME})

In [ ]:
import json
from phase4.run_phase4 import run_selected_pair

PROTOCOL_PATH = REPO_ROOT / "phase4" / "configs" / "controlled_10lang_prior_comparison_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase4_controlled_10lang_prior_comparison")
MATRIX_PATH = Path("/kaggle/working/interference_matrix_10lang.csv")
ORDER_ID = "order-1"  # Run order-1 through order-5.
TRAINING_SEED = 42
METHOD_IDS = ["B2", "B5", "M3"]

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
expected_languages = ["en", "de", "es", "fr", "sw", "ar", "ur", "bg", "ru", "hi"]
if protocol["dataset"]["languages"] != expected_languages:
    raise ValueError("Phase 4 protocol language roster differs from the approved ten-language stream.")
gate = protocol["phase4"]["prior_gate"]
if gate["status"] != "passed" or gate["approved_methods"] != METHOD_IDS or protocol["run_methods"] != METHOD_IDS:
    raise RuntimeError("The approved B2/B5/M3 comparison gate is not configured in the protocol.")
if TRAINING_SEED not in protocol["training"]["seeds"]:
    raise ValueError(f"Seed {TRAINING_SEED} is not configured in the comparison protocol.")
if not METHOD_IDS or len(set(METHOD_IDS)) != len(METHOD_IDS):
    raise ValueError("Select one or more distinct Phase 4 methods.")

if "M3" in METHOD_IDS:
    subprocess.run([
        sys.executable, "-m", "phase4.prepare_interference_matrix",
        "--output", str(MATRIX_PATH),
    ], cwd=REPO_ROOT, check=True)

records = []
for method_id in METHOD_IDS:
    result = run_selected_pair(
        method_id=method_id,
        order_id=ORDER_ID,
        training_seed=TRAINING_SEED,
        protocol_path=PROTOCOL_PATH,
        artifact_root=ARTIFACT_ROOT,
        interference_matrix_path=MATRIX_PATH,
    )
    records.extend(result if isinstance(result, list) else [result])
print("Completed run IDs:", [record["run_id"] for record in records])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from phase2.protocol import build_manifest, load_protocol
from phase2.results import aggregate_results
from phase2.run_baseline import validate_prepared_artifacts

protocol = load_protocol(PROTOCOL_PATH)
prepared = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
manifest = build_manifest(protocol)
manifest["data_artifacts"] = prepared["data_artifacts"]
manifest["data_manifest_sha256"] = prepared["data_manifest_sha256"]
aggregate = aggregate_results(manifest, ARTIFACT_ROOT / "results")
archive_path = Path("/kaggle/working") / f"phase4_{ORDER_ID}_seed_{TRAINING_SEED}_{'_'.join(METHOD_IDS)}.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    archive.write(ARTIFACT_ROOT / "protocol_manifest.json", "protocol_manifest.json")
    for record in records:
        result_path = ARTIFACT_ROOT / "results" / f"{record['run_id']}.json"
        archive.write(result_path, f"results/{result_path.name}")
    archive.writestr("aggregate.json", json.dumps(aggregate, indent=2, sort_keys=True) + "\n")
    if "M3" in METHOD_IDS:
        archive.write(MATRIX_PATH, f"priors/{MATRIX_PATH.name}")
        metadata_path = MATRIX_PATH.with_name(f"{MATRIX_PATH.stem}_metadata.json")
        archive.write(metadata_path, f"priors/{metadata_path.name}")
print("Download and save this Kaggle output:", archive_path)